# Tests de l'API FastAPI

Ce notebook teste l'API déployée en ligne sur le Space Hugging Face.

Endpoints testés :

- `GET /api/health`
- `GET /api/metadata`
- `POST /api/baseline`
- `POST /api/predict`
- `POST /api/recommend`


## Configuration

Le notebook pointe par défaut vers l'URL publique du Space :

- `https://stephmnt-rendement-agricole.hf.space/api`

Si le Space utilise un autre domaine public, il suffit de modifier `API_BASE_URL` dans la cellule suivante.


In [4]:
import pandas as pd
import requests

SPACE_BASE_URL = "https://stephmnt-rendement-agricole.hf.space"
API_BASE_URL = f"{SPACE_BASE_URL}/api"
REQUEST_TIMEOUT_SECONDS = 30
HECTARES = 12.5

print(f"API testée : {API_BASE_URL}")


API testée : https://stephmnt-rendement-agricole.hf.space/api


## Fonctions utilitaires HTTP

Les fonctions ci-dessous interrogent directement l'API publique du Space.


In [5]:
def get_json(endpoint, params=None):
    url = f"{API_BASE_URL}{endpoint}"
    try:
        response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(f"Impossible de joindre l'API en ligne sur {url}: {exc}") from exc
    return response.status_code, response.json()


def post_json(endpoint, payload):
    url = f"{API_BASE_URL}{endpoint}"
    try:
        response = requests.post(url, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(f"Impossible de joindre l'API en ligne sur {url}: {exc}") from exc
    return response.status_code, response.json()


## Tests de disponibilité et métadonnées


In [6]:
health_status, health_response = get_json('/health')

assert health_status == 200
assert health_response['status'] == 'ok'
assert health_response['strategy'] == '2_models_3_predictions_combined'
assert health_response['historical_model_name']
assert health_response['simulation_model_name']

metadata_status, metadata_response = get_json('/metadata')

assert metadata_status == 200
assert metadata_response['countries']
assert metadata_response['available_crops']
assert 'target_year' in metadata_response
assert 'global_reference_profile' in metadata_response
assert 'simulation_options' in metadata_response

preferred_country = 'France' if 'France' in metadata_response['countries'] else metadata_response['countries'][0]

country_metadata_status, country_metadata = get_json('/metadata', params={'country': preferred_country})
assert country_metadata_status == 200
assert country_metadata['available_crops']

preferred_crop = 'Maize' if 'Maize' in country_metadata['available_crops'] else country_metadata['available_crops'][0]

{
    'health': health_response,
    'selected_country': preferred_country,
    'selected_crop': preferred_crop,
    'target_year': country_metadata['target_year'],
    'global_reference_profile': country_metadata['global_reference_profile'],
    'inferred_region': country_metadata.get('inferred_region'),
}


{'health': {'status': 'ok',
  'strategy': '2_models_3_predictions_combined',
  'historical_model_name': 'random_forest_search_01',
  'simulation_model_name': 'linear_regression'},
 'selected_country': 'France',
 'selected_crop': 'Maize',
 'target_year': 2016,
 'global_reference_profile': {'region': 'North',
  'soil_type': 'Sandy',
  'rainfall_mm': 550.2292,
  'temperature_celsius': 27.5095,
  'fertilizer_used': True,
  'irrigation_used': False,
  'weather_condition': 'Sunny',
  'days_to_harvest': 104.0},
 'inferred_region': 'North'}

## Test de `POST /baseline`


In [7]:
baseline_payload = {
    'country': preferred_country,
    'crop': preferred_crop,
}

baseline_status, baseline_response = post_json('/baseline', baseline_payload)

assert baseline_status == 200
assert baseline_response['country'] == preferred_country
assert baseline_response['crop'] == preferred_crop
assert float(baseline_response['p1_historical_prediction']) >= 0.0
assert 'reference_profile' in baseline_response

baseline_response


{'country': 'France',
 'crop': 'Maize',
 'target_year': 2016,
 'p1_historical_prediction': 8.5235,
 'reference_profile': {'region': 'North',
  'soil_type': 'Sandy',
  'rainfall_mm': 867.0,
  'temperature_celsius': 11.01,
  'fertilizer_used': True,
  'irrigation_used': False,
  'weather_condition': 'Sunny',
  'days_to_harvest': 104.0},
 'rainfall_reference_source': 'row_latest_history',
 'temperature_reference_source': 'row_latest_history'}

## Construction des payloads


In [8]:
reference_profile = baseline_response['reference_profile']
simulation_options = country_metadata['simulation_options']

def pick_distinct(options, current_value):
    for option in options:
        if option != current_value:
            return option
    return current_value

user_conditions = {
    'region': pick_distinct(simulation_options['regions'], reference_profile['region']),
    'soil_type': pick_distinct(simulation_options['soil_types'], reference_profile['soil_type']),
    'rainfall_mm': float(reference_profile['rainfall_mm']) + 25.0,
    'temperature_celsius': float(reference_profile['temperature_celsius']) + 1.5,
    'fertilizer_used': not bool(reference_profile['fertilizer_used']),
    'irrigation_used': not bool(reference_profile['irrigation_used']),
    'weather_condition': pick_distinct(simulation_options['weather_conditions'], reference_profile['weather_condition']),
    'days_to_harvest': max(1.0, float(reference_profile['days_to_harvest']) + 7.0),
}

candidate_crops = [crop for crop in ['Maize', 'Wheat', 'Rice, paddy'] if crop in country_metadata['available_crops']]
if not candidate_crops:
    candidate_crops = country_metadata['available_crops'][:3]

predict_payload = {
    'country': preferred_country,
    'crop': preferred_crop,
    'user_conditions': user_conditions,
}

recommend_payload = {
    'country': preferred_country,
    'user_conditions': user_conditions,
    'candidate_crops': candidate_crops,
    'top_n': min(3, len(candidate_crops)),
}

{
    'predict_payload': predict_payload,
    'recommend_payload': recommend_payload,
}


{'predict_payload': {'country': 'France',
  'crop': 'Maize',
  'user_conditions': {'region': 'East',
   'soil_type': 'Chalky',
   'rainfall_mm': 892.0,
   'temperature_celsius': 12.51,
   'fertilizer_used': False,
   'irrigation_used': True,
   'weather_condition': 'Cloudy',
   'days_to_harvest': 111.0}},
 'recommend_payload': {'country': 'France',
  'user_conditions': {'region': 'East',
   'soil_type': 'Chalky',
   'rainfall_mm': 892.0,
   'temperature_celsius': 12.51,
   'fertilizer_used': False,
   'irrigation_used': True,
   'weather_condition': 'Cloudy',
   'days_to_harvest': 111.0},
  'candidate_crops': ['Maize', 'Wheat', 'Rice, paddy'],
  'top_n': 3}}

## Test de `POST /predict`


In [9]:
predict_status, predict_response = post_json('/predict', predict_payload)

assert predict_status == 200
assert predict_response['country'] == preferred_country
assert predict_response['crop'] == preferred_crop
assert float(predict_response['final_prediction']) >= 0.0
assert 'p1_historical_prediction' in predict_response
assert 'local_adjustment' in predict_response
assert 'explanation' in predict_response

{
    'payload': predict_payload,
    'response': {
        'p1_historical_prediction': predict_response['p1_historical_prediction'],
        'local_adjustment': predict_response['local_adjustment'],
        'final_prediction': predict_response['final_prediction'],
    },
}


{'payload': {'country': 'France',
  'crop': 'Maize',
  'user_conditions': {'region': 'East',
   'soil_type': 'Chalky',
   'rainfall_mm': 892.0,
   'temperature_celsius': 12.51,
   'fertilizer_used': False,
   'irrigation_used': True,
   'weather_condition': 'Cloudy',
   'days_to_harvest': 111.0}},
 'response': {'p1_historical_prediction': 8.5235,
  'local_adjustment': -0.1392,
  'final_prediction': 8.3843}}

## Test de `POST /recommend`


In [10]:
recommend_status, recommend_response = post_json('/recommend', recommend_payload)
recommendations = recommend_response['recommendations']

assert recommend_status == 200
assert len(recommendations) > 0
assert recommend_response['best_crop'] == recommendations[0]['crop']
assert recommendations == sorted(
    recommendations,
    key=lambda row: row['final_prediction'],
    reverse=True,
)

recommendations_df = pd.DataFrame(recommendations)
recommendations_df['predicted_total_production_tons'] = recommendations_df['final_prediction'] * HECTARES
recommendations_df[['recommendation_rank', 'crop', 'p1_historical_prediction', 'local_adjustment', 'final_prediction', 'predicted_total_production_tons']]


,recommendation_rank,crop,p1_historical_prediction,local_adjustment,final_prediction,predicted_total_production_tons
0,1,Maize,8.5235,-0.1392,8.3843,104.80375
1,2,Wheat,6.7592,-0.1392,6.6199,82.74875
2,3,"Rice, paddy",4.8313,-0.1392,4.6921,58.65125
